# Dynamics 04c — Coopération payorienne : le FairBot qui se prouve par cadres de Kripke

**Navigation** : [<< 04b-FairBot-Loeb](Dynamics-04b-FairBot-Loeb-Lean.ipynb) | [Version C#](Dynamics-04c-Cooperation-Payorienne-Kripke-CSharp.ipynb) | [Version Rust](Dynamics-04c-Cooperation-Payorienne-Kripke-Rust.ipynb) | [Index](README.md) | [Dynamics-05-Agents-Bornes-Python >>](Dynamics-05-Agents-Bornes-Python.ipynb)

Ce carnet sert la tranche T-2 du plan de consolidation de l'[issue #26985](https://github.com/jsboige/CoursIA/issues/26985), née du Concern de l'issue #26954 : l'état de l'art ouvert de la théorie des jeux (OSGT) n'était cité nulle part dans le dépôt. Il reprend la **coopération payorienne** du post d'Alignment Forum *« Payorian cooperation is easy with Kripke frames »* (transhumanist_atom_understander, 9 mars 2026), lu dans sa copie archivée au gisement bibliographique (`GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames - Alignment Forum LaCP6WyNzX8kiZn3w.pdf`).

Le carnet fait trois choses, et une seule de chaque :

1. **énonce** le FairBot payorien `P ↔ □(□P → Q)` et le lemme de Payor, en citant la source primaire — aucune preuve formelle n'est refaite ici ;
2. **implémente** le bot dans l'esprit du moteur jouet de [Dynamics-04](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ;
3. **mesure** la profondeur de réfutation par cadres de Kripke, avec un moteur déclaré en **copie pédagogique** du laboratoire [Tweety-03b](../../SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python.ipynb) §3.

La frontière mesure/preuve est tenue partout : les réfutations sont des **calculs bornés**, pas des théorèmes ; le côté formel vit dans [`ProgramGames.FairBot`](../game_theory_lean/ProgramGames/FairBot.lean) (niveau L2) et dans [Dynamics-04b](Dynamics-04b-FairBot-Loeb-Lean.ipynb) pour la partie löbienne.


## 1. Le tournoi et la question

Le tournoi de MIRI est un dilemme du prisonnier **one-shot** où chaque programme lit le code de l'autre — ou plutôt une description de son comportement en logique de la prouvabilité de Gödel-Löb (GL). FairBot est défini par « je coopère avec toi si tu coopères avec moi, prouvablement » :

$$F(X) \leftrightarrow \Box\, X(F)$$

Malgré la circularité apparente, deux FairBots coopèrent : c'est la **coopération löbienne**, qui passe par le théorème de Löb.

Critch a proposé une autre voie, le **lemme de Payor** : au lieu de vérifier que l'adversaire coopère, on vérifie que *sa propre coopération prouvable implique* celle de l'adversaire :

$$P \leftrightarrow \Box(\Box P \to Q)$$

avec $P \equiv F(X)$ et $Q \equiv X(F)$. La question de ce carnet : **pourquoi cette forme se réfute-t-elle plus facilement sur un cadre de Kripke ?** Le post répond « deux mondes au lieu de quatre » ; nous allons le mesurer.

Les paiements sont ceux du moteur de Dynamics-04.

In [1]:
# --- Paiements du dilemme du prisonnier (meme convention que Dynamics-04) ---
T, R, P_PAIEMENT, S = 5, 3, 1, 0
assert T > R > P_PAIEMENT > S, "parametres PD non canoniques"
assert 2 * R > T + S, "cooperation mutuelle > alternance"

def payoff_self(mon_action, action_adverse):
    if mon_action == "C":
        return R if action_adverse == "C" else S
    return T if action_adverse == "C" else P_PAIEMENT

print("self  other  payoff")
for me in ("C", "D"):
    for other in ("C", "D"):
        print(f"  {me}     {other}     {payoff_self(me, other)}")

self  other  payoff
  C     C     3
  C     D     0
  D     C     5
  D     D     1


### Lecture : la matrice tient la structure du dilemme

La sortie donne exactement quatre paiements — `C/C → 3`, `C/D → 0`, `D/C → 5`, `D/D → 1` — et les deux assertions en tête de cellule fixent l'ordre strict `T > R > P > S` ainsi que la préférence pour la coopération mutuelle sur l'alternance. C'est cette matrice que les bots ci-dessous vont jouer, sans itération ni réputation : tout se décide avant le coup, sur la seule lecture du code adverse.

## 2. Le lemme de Payor, et ce que ce carnet n'en fait pas

**Lemme de Payor.** Pour toute formule $A$ et tout $B$ : $\vdash \Box(\Box A \to B) \to A$.

L'attribution est double, et le post la donne telle quelle : James Payor raconte dans le fil être arrivé à `□(□A→B)→A` en raisonnant sur les étapes de l'algorithme, et c'est **Andrew Critch** qui propose de s'en servir à la place du théorème de Löb pour prouver la coopération de soi — « *Instead of Löb's theorem, we use what he calls "Payor's lemma"* ». Le nom est resté au lemme, l'usage au post.

La preuve est **arithmétique** — elle dérive du théorème de Löb — et n'est pas refaite ici : le carnet travaille exclusivement du côté **sémantique**, où le lemme se lit comme une contrainte sur les cadres. L'axiome de Löb $\Box(\Box A \to A) \to \Box A$ y est valide exactement sur les cadres transitifs et **bien fondés**.

Déclarations de provenance :

- **copie pédagogique** : le moteur de Kripke de la cellule suivante est une copie déclarée du §3 du laboratoire Tweety-03b (mêmes classes, même clause de forcing), motif = un carnet est autonome et ne peut pas importer un autre carnet ;
- **citation** : l'énoncé du lemme et les dérivations du post sont cités, jamais re-démontrés — le côté mécanisé vit dans le lake `game_theory_lean` (niveau L2) et, pour la partie löbienne, dans le carnet 04b ;
- **archive** : la copie du post lue pour ce carnet est celle du gisement bibliographique, pas une page web revisitée.

In [2]:
# --- Moteur de Kripke : AST gelee + forcing recursif ---
# COPIE PEDAGOGIQUE DECLAREE du §3 de Tweety-03b-Modal-Lab-Lean-Python.ipynb :
# memes dataclasses, meme clause de forcing. Motif : un carnet est autonome.
from dataclasses import dataclass

@dataclass(frozen=True)
class Atome:   nom: str
@dataclass(frozen=True)
class Non:     f: object
@dataclass(frozen=True)
class Imp:     a: object; b: object
@dataclass(frozen=True)
class Et:      a: object; b: object
@dataclass(frozen=True)
class Boite:   f: object
@dataclass(frozen=True)
class Diamant: f: object

@dataclass(frozen=True)
class Modele:
    mondes: tuple        # de mondes (entiers)
    relation: frozenset  # de couples (x, y) : x voit y
    valuation: dict      # Atome.nom -> frozenset des mondes ou l'atome est vrai

    def successeurs(self, x):
        return frozenset(y for (a, y) in self.relation if a == x)

def satisfait(M, x, f):
    """Forcing x |= f : Boite = tous les successeurs, Diamant = au moins un."""
    if isinstance(f, Atome):   return x in M.valuation[f.nom]
    if isinstance(f, Non):     return not satisfait(M, x, f.f)
    if isinstance(f, Imp):     return (not satisfait(M, x, f.a)) or satisfait(M, x, f.b)
    if isinstance(f, Et):      return satisfait(M, x, f.a) and satisfait(M, x, f.b)
    if isinstance(f, Boite):   return all(satisfait(M, y, f.f) for y in M.successeurs(x))
    if isinstance(f, Diamant): return any(satisfait(M, y, f.f) for y in M.successeurs(x))
    raise TypeError(f"formule inconnue : {f!r}")

def show(f):
    if isinstance(f, Atome):   return f.nom
    if isinstance(f, Non):     return "non " + show(f.f)
    if isinstance(f, Imp):     return f"({show(f.a)} -> {show(f.b)})"
    if isinstance(f, Et):      return f"({show(f.a)} et {show(f.b)})"
    if isinstance(f, Boite):   return "[]" + show(f.f)
    if isinstance(f, Diamant): return "<>" + show(f.f)

# Mini-test : le monde 0 voit le monde 1 ; P est vrai en 1 seulement.
m_test = Modele((0, 1), frozenset({(0, 1)}), {"P": frozenset({1})})
verdicts = {
    "P en 0":      satisfait(m_test, 0, Atome("P")),
    "[]P en 0":    satisfait(m_test, 0, Boite(Atome("P"))),
    "[]P -> P en 0": satisfait(m_test, 0, Imp(Boite(Atome("P")), Atome("P"))),
    "<>P en 1":    satisfait(m_test, 1, Diamant(Atome("P"))),
}
for etiquette, valeur in verdicts.items():
    print(f"  {etiquette:14s} : {'vrai' if valeur else 'faux'}")
assert verdicts["[]P en 0"] and not verdicts["[]P -> P en 0"]
print("\nMoteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.")

  P en 0         : faux
  []P en 0       : vrai
  []P -> P en 0  : faux
  <>P en 1       : faux

Moteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.


### Lecture : le monde mort et la vacuité

`[]P en 0` est vrai alors que `[]P -> P en 0` est faux — la sortie montre les deux lignes opposées, et l'assertion finale le confirme. Le monde 1, sans successeur, rend `[]` vacuement vrai ; c'est exactement la clause qui portera la réfutation : une boîte fausse signale toujours un monde *visible* où son contenu échoue.

## 3. Réfuter en empilant des contraintes sur un cadre

La méthode du post est un raisonnement par l'absurde sémantique. Pour prouver que FairBot coopère, on suppose qu'il **défecte** : $\neg P$. Comme $P \leftrightarrow \Box(\Box P \to Q)$, cela donne

$$\neg\Box(\Box P \to Q) \;\equiv\; \Diamond(\Box P \land \neg Q)$$

c'est-à-dire un monde visible $w_1$ où $\Box P$ et $\neg Q$ tiennent. On propage ensuite les boîtes le long de la visibilité — transitivité : si $w_0$ voit $w_1$ et $w_1$ voit $w_2$, alors $w_0$ voit $w_2$ — et l'on cherche une **contradiction locale**, un atome et sa négation au même monde.

La cellule suivante code cet algorithme : un dérouleur qui, à chaque obligation de possibilité non satisfaite, ouvre **un** monde de plus (chaîne croissante), propage les nécessités, et s'arrête sur contradiction. Deux limites sont assumées d'emblée : la borne de profondeur rend un statut INCOMPLET plutôt que « cohérent », et le dérouleur n'encode **pas** la bonne-fondation des cadres de GL — la section 5 montrera précisément ce que cela change pour le côté löbien.

In [3]:
# --- Derouleur de refutation : saturer des contraintes jusqu'a contradiction ---
# L'algorithme du post, borne en profondeur. Ce n'est PAS un prouveur de GL :
# au-dela de la borne, une obligation ouverte rend INCOMPLET, jamais "coherent".

def A(nom):   return Atome(nom)
def non(f):   return Non(f)
def imp(a, b): return Imp(a, b)
def box(f):   return Boite(f)
def dia(f):   return Diamant(f)
def equiv(f, g):
    return Et(Imp(f, g), Imp(g, f))

class Contradiction(Exception):
    def __init__(self, monde, f):
        self.monde, self.f = monde, f
        super().__init__(f"monde {monde} : {show(f)} et sa negation")

def voit(i, j):
    # chaine transitive : le monde i voit tout monde cree apres lui
    return i < j

def _ajoute(mondes, w, f):
    if Non(f) in mondes[w]:
        raise Contradiction(w, f)
    if isinstance(f, Non) and f.f in mondes[w]:
        raise Contradiction(w, f.f)
    mondes[w].add(f)

def refute(premisses, max_profondeur=6):
    """Rend ("unsat", monde) | ("sat", mondes) | ("incomplet", None)."""
    mondes = {0: set(premisses)}
    try:
        while True:
            change = False
            for w in sorted(list(mondes)):
                for f in sorted(list(mondes[w]), key=show):
                    if isinstance(f, Non) and isinstance(f.f, Non):
                        if f.f.f not in mondes[w]:
                            _ajoute(mondes, w, f.f.f); change = True
                    elif isinstance(f, Et):
                        for g in (f.a, f.b):
                            if g not in mondes[w]:
                                _ajoute(mondes, w, g); change = True
                    elif isinstance(f, Imp):
                        if f.a in mondes[w] and f.b not in mondes[w]:
                            _ajoute(mondes, w, f.b); change = True
                        if Non(f.b) in mondes[w] and Non(f.a) not in mondes[w]:
                            _ajoute(mondes, w, Non(f.a)); change = True
                    elif isinstance(f, Non) and isinstance(f.f, Imp):
                        for g in (f.f.a, Non(f.f.b)):
                            if g not in mondes[w]:
                                _ajoute(mondes, w, g); change = True
                    elif isinstance(f, Boite):
                        for v in [x for x in mondes if voit(w, x)]:
                            if f.f not in mondes[v]:
                                _ajoute(mondes, v, f.f); change = True
                    elif isinstance(f, Non) and isinstance(f.f, Diamant):
                        for v in [x for x in mondes if voit(w, x)]:
                            if Non(f.f.f) not in mondes[v]:
                                _ajoute(mondes, v, Non(f.f.f)); change = True
                    elif isinstance(f, Non) and isinstance(f.f, Boite):
                        besoin = Non(f.f.f)
                        if not any(besoin in mondes[v] for v in mondes if voit(w, v)):
                            if max(mondes) >= max_profondeur:
                                return ("incomplet", None)
                            mondes[max(mondes) + 1] = {besoin}; change = True
                    elif isinstance(f, Diamant):
                        besoin = f.f
                        if not any(besoin in mondes[v] for v in mondes if voit(w, v)):
                            if max(mondes) >= max_profondeur:
                                return ("incomplet", None)
                            mondes[max(mondes) + 1] = {besoin}; change = True
            if not change:
                return ("sat", mondes)
    except Contradiction as c:
        return ("unsat", c)

print("Derouleur defini : refute(premisses) -> (statut, info).")
print("Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).")

Derouleur defini : refute(premisses) -> (statut, info).
Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).


### Lecture du dérouleur

Trois statuts, et aucun des trois n'est un « vrai » :

| statut | sens | ce qu'on peut en dire |
|---|---|---|
| `unsat` | l'atome et sa négation se sont rencontrés au monde `info.monde` | **réfutation** : l'hypothèse de départ est incompatible avec les prémisses |
| `sat` | saturation sans contradiction | un **modèle fini** des prémisses : le comportement observé est *possible*, pas *forcé* |
| `incomplet` | borne atteinte, obligation encore ouverte | ni cohérent, ni contradictoire : **on ne sait pas** |

Deux limites à garder en tête avant de lire le tableau suivant.

1. **`sat` n'est pas une preuve de cohérence.** Le modèle trouvé est un modèle du *cadre employé* — une chaîne transitive finie de mondes. Un autre cadre pourrait fermer la réfutation ; un modèle, c'est une possibilité, pas un théorème.
2. **Le dérouleur n'encode pas la bonne-fondation.** Les cadres de GL sont transitifs **et** bien fondés : aucune chaîne infinie de mondes accessibles. Ici la visibilité est la simple chaîne `i < j` sur les mondes déjà créés, sans aucun axiome de bonne-fondation. Quand une obligation ouvre un monde de plus à chaque passe, le dérouleur en ouvre indéfiniment et bute sur sa borne. C'est une propriété du **dérouleur**, pas une découverte sur Löb — et c'est précisément ce qui rend le cas löbien `INCOMPLET` à la section suivante.

In [4]:
# --- Les quatre jeux de premisses du post ---
P_AT, Q_AT, H_AT = A("P"), A("Q"), A("H")

COND_PAYOR   = equiv(P_AT, box(imp(box(P_AT), Q_AT)))   # P coopere ssi P prouve []([]P -> Q)
COND_PAYOR_Q = equiv(Q_AT, box(imp(box(Q_AT), P_AT)))   # la meme definition, vue de Q
COND_COOP    = box(Q_AT)                                # CooperateBot : coopere toujours
COND_DEFECT  = box(non(Q_AT))                           # DefectBot : trahit toujours
COND_LOEB    = equiv(H_AT, box(H_AT))                   # phrase de Henkin : H <-> []H

# Pourquoi les definitions sont AUSSI posees sous une boite (box(COND_...)) :
# la refutation de ~P se joue dans le monde w1, ou l'on simule l'adversaire.
# Pour que cette simulation raisonne avec SA definition, celle-ci doit etre
# accessible dans les successeurs -- c'est la premisse supplementaire que le
# fil reclame explicitement ("repeating the description of myself but wrapped
# in a provability modality so it's accessible in the simulations", 8mo).

CAS = {
    "P vs CooperateBot": [COND_PAYOR, box(COND_PAYOR), COND_COOP, non(P_AT)],
    "P vs Payorian Q":   [COND_PAYOR, box(COND_PAYOR),
                          COND_PAYOR_Q, box(COND_PAYOR_Q), non(P_AT)],
    "P vs DefectBot":    [COND_PAYOR, box(COND_PAYOR), COND_DEFECT, non(P_AT)],
    "Lob : H <-> []H":   [COND_LOEB, box(COND_LOEB), non(H_AT)],
}

def profondeur_minimale(premisses, borne=6):
    """Plus petite borne qui ferme ; (None, 'incomplet', None) si aucune ne ferme."""
    for k in range(1, borne + 1):
        statut, info = refute(premisses, k)
        if statut != "incomplet":
            return k, statut, info
    return None, "incomplet", None

print(f"{'jeu de premisses':<20} {'statut':<10} {'borne':<7} detail")
print("-" * 78)
for nom, premisses in CAS.items():
    borne, statut, info = profondeur_minimale(premisses)
    if statut == "unsat":
        detail = f"contradiction en {info.monde} : {show(info.f)}"
    elif statut == "sat":
        detail = f"modele a {len(info)} mondes"
    else:
        detail = "borne atteinte, obligation ouverte"
    print(f"{nom:<20} {statut:<10} {str(borne):<7} {detail}")

# --- Controles : les trois profondeurs annoncees par le post, plus la frontiere ---
b1, s1, _ = profondeur_minimale(CAS["P vs CooperateBot"])
b2, s2, _ = profondeur_minimale(CAS["P vs Payorian Q"])
b3, s3, m3 = profondeur_minimale(CAS["P vs DefectBot"])
b4, s4, _ = profondeur_minimale(CAS["Lob : H <-> []H"])

assert (s1, b1) == ("unsat", 1), (s1, b1)   # un seul monde ajoute
assert (s2, b2) == ("unsat", 2), (s2, b2)   # deux mondes : c'est le "2 au lieu de 4"
assert s3 == "sat" and len(m3) == 2, (s3, m3)
assert s4 == "incomplet", s4

# Lecture du modele de P vs DefectBot : le monde 1 porte []P et ~Q.
# L'atome P n'y figure pas -- c'est la biconditionnelle, verifiee dans la
# cellule suivante, qui l'y rend vrai.
assert Boite(P_AT) in m3[1] and Non(Q_AT) in m3[1]
print()
print("P vs DefectBot : monde 1 =", sorted(show(f) for f in m3[1]))
print("Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.")

jeu de premisses     statut     borne   detail
------------------------------------------------------------------------------
P vs CooperateBot    unsat      1       contradiction en 1 : Q
P vs Payorian Q      unsat      2       contradiction en 2 : P
P vs DefectBot       sat        1       modele a 2 mondes
Lob : H <-> []H      incomplet  None    borne atteinte, obligation ouverte

P vs DefectBot : monde 1 = ['((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))', '(P -> []([]P -> Q))', '([]([]P -> Q) -> P)', '[]P', 'non ([]P -> Q)', 'non Q']
Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.


### Vérifier un `sat` : les deux moteurs doivent s'accorder

Un `sat` n'est pas une ligne de console, c'est l'affirmation qu'un modèle existe. On la vérifie en reconstruisant un `Modele` à partir des mondes saturés, puis en repassant **chaque prémisse** au moteur de forcing de la section 2 — un moteur écrit indépendamment, pour un autre carnet.

La reconstruction prend une convention, et il faut la nommer : un atome est vrai dans un monde **sauf si** sa négation y figure. C'est une fermeture par défaut, pas une conséquence du dérouleur. Si les prémisses ne tiennent pas sous cette convention, le `sat` est faux et l'`assert` le signale ; si elles tiennent, on a bien un modèle, et les deux moteurs s'accordent.

In [5]:
# --- Un "sat" se verifie : on repasse le modele au moteur de la section 2 ---
def atomes(f):
    if isinstance(f, Atome):            return {f.nom}
    if isinstance(f, Non):              return atomes(f.f)
    if isinstance(f, (Imp, Et)):        return atomes(f.a) | atomes(f.b)
    if isinstance(f, (Boite, Diamant)): return atomes(f.f)
    return set()

def vers_modele(mondes):
    """Modele reconstruit des mondes satures.

    Fermeture par defaut : un atome est vrai en w sauf si non(Atome) y figure.
    La relation est la chaine transitive i < j, celle du derouleur.
    """
    noms = set()
    for fs in mondes.values():
        for f in fs:
            noms |= atomes(f)
    valuation = {n: frozenset(w for w in mondes if Non(Atome(n)) not in mondes[w])
                 for n in noms}
    relation = frozenset((i, j) for i in mondes for j in mondes if i < j)
    return Modele(tuple(sorted(mondes)), relation, valuation)

M3 = vers_modele(m3)
premisses = CAS["P vs DefectBot"]
verdicts = [satisfait(M3, 0, p) for p in premisses]
for p, v in zip(premisses, verdicts):
    print(f"  {'vrai' if v else 'FAUX':4s}  {show(p)}")
assert all(verdicts), "le modele ne satisfait pas les premisses : le 'sat' serait faux"

print()
print("Modele valide : les quatre premisses tiennent en w0.")
print("P est vrai en w1 par la biconditionnelle, pas par propagation de []P :")
print("le monde 1 est terminal, donc []P n'y contraint aucun successeur.")

  vrai  ((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))
  vrai  []((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))
  vrai  []non Q
  vrai  non P

Modele valide : les quatre premisses tiennent en w0.
P est vrai en w1 par la biconditionnelle, pas par propagation de []P :
le monde 1 est terminal, donc []P n'y contraint aucun successeur.


### Lecture du tableau

**Une prémisse qui n'est pas un détail.** Les deux définitions sont posées à la racine **et** sous une boîte. Sans la boîte, la définition de l'adversaire n'atteint pas `w1`, la simulation ne sait pas quel bot elle simule, et la réfutation rend un `sat` trompeur. Le fil le formule lui-même : *repeating the description of myself but wrapped in a provability modality so it's accessible in the simulations*. C'est une prémisse du modèle, pas un ajustement du code — et c'est le genre d'oubli qui fait passer un carnet pour une preuve.

**P vs CooperateBot — un monde.** `□Q` à la racine place `Q` dans le seul monde créé, où `¬Q` était déjà posé par la négation de la clause de Payor. La contradiction est **interne** au monde 1 : aucun enchaînement n'est nécessaire.

**P vs Payorian Q — deux mondes.** C'est le résultat central. En `w1` on a `□P` et `¬Q`. Le `¬Q` attaque la définition de Q, qui ouvre `w2` avec `¬P` ; et le `□P` de `w1` place `P` dans `w2`. La contradiction naît **entre les deux définitions**, dans un monde que chacune atteint par un chemin différent. Deux mondes ajoutés, pas quatre.

**P vs DefectBot — saturé, deux mondes.** Le modèle est petit : `w0` porte `¬P`, `w1` porte `□P` et `¬Q`. Tout tient, et le point qui fait tenir est que **`w1` ne voit rien** : `□(□P→Q)` y est vacuement vrai, comme la boîte sur un monde terminal de la section 2. Ce `sat` n'est pas un accident de programmation : la cellule suivante le **repasse au moteur de la section 2** et vérifie que les quatre prémisses tiennent. C'est la **frontière de l'hypothèse de cohérence** — la réfutation seule ne conclut pas que le FairBot payorien trahit contre DefectBot. Il faut lui ajouter une hypothèse de cohérence sur son propre système de preuve, et c'est écrit dans le post comme une réserve, pas comme un théorème.

**Löb — incomplet.** La phrase de Henkin `H ↔ □H` n'ouvre pas une contradiction : elle ouvre un monde de plus à chaque passe, indéfiniment. Le dérouleur rend `incomplet`, ce qui est la réponse **honnête** de cet instrument-là. Un prouveur de GL fermerait, parce qu'il dispose de l'axiome de Löb et de la bonne-fondation ; notre dérouleur ne les a pas.

## 4. Le tournoi, en jouets

La section 3 a réfuté des formules. Il reste à voir ce que ces réfutations **disent du jeu**. On reprend la convention du carnet `Dynamics-04-Open-Source-Game-Theory-Python` — un bot rend `"C"` ou `"D"` après avoir lu quelque chose de son adversaire — en remplaçant la lecture du code source par une **déclaration** en clair : chaque bot publie sa règle, et c'est ce texte que les autres lisent. Une caricature assumée, et il faut dire exactement ce qu'elle vaut :

- le tournoi **illustre** les issues que l'analyse de Kripke contraint ;
- il ne les **démontre pas**. Sur le cas `P vs DefectBot`, l'analyse rend `sat` : deux issues sont compatibles avec les prémisses, et le jouet en choisit une par construction. Ce choix n'est pas une dérivation, c'est une règle d'écriture.

La déclaration joue le rôle de la définition de l'adversaire dans une simulation modale : le bot ne voit pas la fonction, il voit ce que l'autre annonce de lui-même — exactement la limite du dérouleur de la section 3, qui ne connaît de l'adversaire que la formule qu'on lui donne.

In [6]:
# --- Les trois bots, chacun declarant sa regle ---
# Un bot lit la DECLARATION de son adversaire (le jouet ne voit que le texte,
# pas la fonction : c'est la caricature assumee du paragraphe precedent).

def coopere_toujours(_decl):
    return "C"

def trahit_toujours(_decl):
    return "D"

def fairbot_payorien(decl):
    # Coopere si la declaration adverse garantit la cooperation contre
    # un cooperateur conditionnel -- c'est la clause de Payor, en texte.
    if "coopere toujours" in decl or "coopere si l'adversaire" in decl:
        return "C"
    return "D"

# nom -> (decision, declaration que les autres lisent)
BOTS = {
    "cooperatebot":     (coopere_toujours,  "coopere toujours"),
    "defectbot":        (trahit_toujours,   "trahit toujours"),
    "fairbot_payorien": (fairbot_payorien,  "coopere si l'adversaire declare coopere"),
}

def gain(mien, sien):
    if mien == "C" and sien == "C":
        return 3
    if mien == "C":
        return 0
    return 5 if sien == "C" else 1

def jouer(nom_a, nom_b):
    fa, decl_a = BOTS[nom_a]
    fb, decl_b = BOTS[nom_b]
    return fa(decl_b), fb(decl_a)

# --- Matrice des gains ---
noms = list(BOTS)
print(f"{'':<18}" + "".join(f"{n:>18}" for n in noms))
print("-" * (18 + 18 * len(noms)))
for a in noms:
    ligne = []
    for b in noms:
        ca, cb = jouer(a, b)
        ligne.append(f"{ca}/{cb} {gain(ca, cb)}")
    print(f"{a:<18}" + "".join(f"{c:>18}" for c in ligne))

# --- Ce que la section 3 contraint, et ce qu'elle ne contraint pas ---
assert jouer("fairbot_payorien", "cooperatebot") == ("C", "C")      # ferme en 1 monde
assert jouer("fairbot_payorien", "fairbot_payorien") == ("C", "C")  # ferme en 2 mondes

ca, cb = jouer("fairbot_payorien", "defectbot")
print()
print(f"P vs DefectBot (jouet) : {ca}/{cb} -> gain {gain(ca, cb)}")
print("L'analyse de la section 3 rend 'sat' ici : le jouet choisit,")
print("il ne derive pas. Les deux assertions ci-dessus, elles, sont contraintes.")

                        cooperatebot         defectbot  fairbot_payorien
------------------------------------------------------------------------
cooperatebot                   C/C 3             C/D 0             C/C 3
defectbot                      D/C 5             D/D 1             D/D 1
fairbot_payorien               C/C 3             D/D 1             C/C 3

P vs DefectBot (jouet) : D/D -> gain 1
L'analyse de la section 3 rend 'sat' ici : le jouet choisit,
il ne derive pas. Les deux assertions ci-dessus, elles, sont contraintes.


### Lecture du tournoi

Diagonale : `(3,3)`, `(1,1)`, `(3,3)`. Le FairBot payorien coopère contre le coopérateur inconditionnel et contre lui-même — les deux seules cases que la section 3 **contraint**, et les deux `assert` du code le vérifient.

La case qui instruit est `P vs DefectBot`. Le jouet rend `(1,1)`, et c'est exactement le piège : on serait tenté de lire ce `1/1` comme « le FairBot payorien trahit contre DefectBot, donc il est prudent ». La section 3 dit autre chose — `sat`, deux mondes — et la différence n'est pas une nuance :

- le jouet **applique une règle de correspondance textuelle** ; il n'a pas de notion de preuve, donc rien ne l'empêche de rendre `D` ;
- l'analyse de Kripke **laisse les deux issues ouvertes** : le modèle à deux mondes est un modèle *des prémisses*, il ne prouve pas que le bot joue `C` en vrai ;
- conclure « il trahit » demande une hypothèse supplémentaire, la cohérence du système de preuve du bot. Le post la pose explicitement ; beaucoup de relectures l'oublient.

Autrement dit : le tournoi et la réfutation ne sont pas deux mesures de la même chose. Le tournoi **exhibe** un comportement ; la réfutation **borne** ce qui est forcé. Confondre les deux, c'est prendre un modèle pour un théorème.

## 5. Deux mondes au lieu de quatre

Le titre du post est une comparaison : la clause de Payor réfute en **deux** mondes là où la route löbienne en demande **quatre**. Il faut séparer ce que ce carnet a mesuré de ce qu'il ne fait que citer, parce que la moitié citée est la moitié qui compte pour le titre.

| Ce qui est en jeu | Statut ici | Détail |
|---|---|---|
| Réfutation de `¬P` sous la clause de Payor, contre un Payorien | **mesuré** — 1 monde contre CooperateBot, 2 contre un Payorien | sections 3 et 4, `assert` du carnet |
| Réfutation de `¬P` contre DefectBot | **mesuré** — `sat`, modèle à 2 mondes | frontière de l'hypothèse de cohérence |
| Route löbienne : « quatre mondes » | **cité**, non re-mesuré | le dérouleur rend `incomplet` sur la phrase de Henkin seule |

Pourquoi le « quatre » ne se re-mesure pas avec cet instrument. La route löbienne part de `F(X) ↔ □(X(F))`, donc pour `F` contre `F` il faut établir un point fixe : `F(F)=C` ssi `□(F(F)=C)`. La preuve du post introduit une **phrase de Henkin** `H ↔ □H` et une **auxiliaire** `L ↔ (□L → H)` ; c'est cette auxiliaire qui rend la dérivation finie, et elle la rend finie **parce qu'elle est une instance de l'axiome de Löb** : `□(□L→H) → L`-style, appliqué à un antécédent qu'on vient de dériver. L'axiome de Löb encode la bonne-fondation des cadres de GL.

Notre dérouleur n'a ni l'axiome ni la bonne-fondation : il empile des contraintes de boîte et ouvre un monde de plus à chaque fois qu'une obligation reste ouverte. Sur `H ↔ □H` avec `¬H`, il en ouvre indéfiniment — d'où l'`incomplet` de la section 3. **Ce n'est pas un échec de la réfutation, c'est la limite de l'instrument**, et la dire fait partie du résultat.

Ce que la comparaison garde de solide, et qui est le vrai contenu pédagogique : les deux routes ne coûtent pas la même chose parce qu'elles ne prouvent pas la même chose. La clause de Payor ajoute une **boîte en position d'antécédent** — `□P` sous le `□` extérieur — et cette boîte intérieure est ce qui referme la réfutation en descendant une seule fois. Lire `□(□P→Q)` comme « je coopère si je peux prouver que ma coopération implique la tienne » est exactement ce qui rend la preuve courte : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement celle de l'adversaire.

## 6. Les variantes du fil — citées, pas re-mesurées

La discussion qui suit le post propose trois bots de plus. Aucun n'est implémenté ici, et c'est un choix : leurs définitions sortent du fragment que le dérouleur traite (voir l'avertissement en fin de section). Les définitions sont citées telles quelles.

**PrudentBot d'origine** — `P(X) ↔ □(X(P) ∧ ◊¬X(DB))` : il coopère s'il prouve que l'adversaire coopère avec lui **et** qu'il existe un monde où cet adversaire trahit contre DefectBot. L'auteur du post juge lui-même cette référence à DefectBot « un peu bricolée » (*a little hacked*), et c'est ce qui motive la suite.

**SharkBot** — `S(X) ↔ □(□S(X) → X(S)) ∧ ◊(X(S) → ◊S(X))` : la clause de Payor, plus une seconde condition sur les diamants. L'auteur annonce « je suis assez sûr » (*I'm pretty sure*) qu'il coopère avec lui-même et avec FairBot, et qu'il trahit contre CooperateBot. **« Assez sûr » n'est pas un théorème** : c'est une conjecture de discussion, et ce carnet ne la vérifie pas.

**PrudentBot payorien** — `P ↔ □(□P→Q) ∧ □(□¬P→¬Q)`, obtenu en simulant l'adversaire deux fois : une fois sous l'hypothèse d'une preuve que je coopère, une fois sous l'hypothèse d'une preuve que je défais. La réfutation part de `¬P` et rend `◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` : **deux** jeux de contraintes à réfuter, un par simulation. L'auteur conclut lui-même : « je ne suis pas vraiment sûr de comment tout cela va tourner » (*I'm not really sure how any of this will turn out*).

### Pourquoi ces variantes ne passent pas dans le dérouleur

`◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` contient une **disjonction**, et `¬P` sur `P ≡ A∧B` produit `¬A ∨ ¬B`. Or le dérouleur de la section 3 ne décompose **ni les disjonctions ni les négations de conjonction** : il ne connaît que `¬¬`, `∧`, `→`, `¬(A→B)`, `□`, `¬◊`, `¬□` et `◊`. Une formule qu'il n'a pas décomposée reste dans le monde sans être traitée, et le dérouleur rendrait `sat` — **un `sat` qui ne voudrait rien dire**. C'est la raison pour laquelle les quatre cas de la section 3 ont été choisis sans disjonction : sur ce fragment-là, le résultat est fiable ; en dehors, il ne l'est pas.

Cette limite n'est pas propre au code. James Payor note dans le fil que la traduction `◊ = ¬□¬` **ne reste pas fidèle à l'intention** (*the "not box not" translation doesn't actually stay very faithful to the intent*), et que la manipulation des longueurs de preuve y est délicate. Le dérouleur implémente exactement cette traduction-là : quand il rencontre `¬□φ`, il ouvre un monde avec `¬φ`. C'est commode, c'est mesurable, et ce n'est pas la même chose que « il existe un monde où φ est vrai » au sens de la logique de la prouvabilité.

## 7. Exercices

Trois exercices, du plus mécanique au plus conceptuel. Les cellules sont des squelettes : elles s'exécutent telles quelles et rendent `None` tant qu'elles ne sont pas complétées. Aucune ne lève d'erreur volontairement.

**Exercice 1 — réfuter avec des disjonctions.** Le dérouleur ne décompose pas `¬(A∧B)`, qui vaut `¬A ∨ ¬B`. Écrire un `refute_branche` qui, en présence d'une disjonction à satisfaire, **explore les deux branches** et ne déclare `unsat` que si toutes les branches se ferment. Le tester sur les quatre cas de la section 3 : il doit rendre exactement les mêmes statuts.

In [7]:
# Exercice 1 — refutation avec disjonctions
def refute_branche(premisses, max_profondeur=6):
    """Comme refute, mais decompose les disjonctions en explorant les deux branches.

    Retour attendu :
      ("unsat", None)      si TOUTES les branches se ferment
      ("sat", mondes)      si une branche sature sans contradiction
      ("incomplet", None)  si une branche atteint la borne
    """
    resultat = None  # TODO etudiant : explorer les deux branches, n'accepter
    #                  unsat que si toutes se ferment
    return resultat

print("Exercice 1 — refute_branche reste a completer.")
print("Controle attendu une fois complete : les quatre cas de la section 3")
print("doivent rendre exactement les memes (statut, borne) que le tableau.")

Exercice 1 — refute_branche reste a completer.
Controle attendu une fois complete : les quatre cas de la section 3
doivent rendre exactement les memes (statut, borne) que le tableau.


**Exercice 2 — le PrudentBot payorien.** Construire `COND_PRUDENT` à partir de la définition citée en section 6, réfuter `¬P` avec le dérouleur **sans** branchement, puis avec `refute_branche`. Expliquer en trois phrases pourquoi le premier résultat ne répond pas à la question.

In [8]:
# Exercice 2 — le PrudentBot payorien
# Definition citee en section 6 :   P <-> []([]P -> Q) /\ []([]~P -> ~Q)
COND_PRUDENT = None  # TODO etudiant : construire la formule

# Puis, pour le meme jeu [COND_PRUDENT, non(P_AT)] :
#   statut_sans  = refute(...)            -> que rend le derouleur, et pourquoi ce n'est pas une reponse ?
#   statut_avec  = refute_branche(...)    -> ce que donne l'exploration des deux branches
resultat_ex2 = None  # TODO etudiant

print("Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.")
print("Rappel : la negation de ~P developpe deux branches, []P et ~Q d'une part,")
print("         []~P et Q de l'autre.")

Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.
Rappel : la negation de ~P developpe deux branches, []P et ~Q d'une part,
         []~P et Q de l'autre.


**Exercice 3 — compter les mondes.** Sur le cas löbien, relever le nombre de mondes créés pour chaque borne de 1 à 6. En déduire pourquoi aucune borne ne rendra ce cas `sat` ni `unsat`, et ce qu'il faudrait ajouter au dérouleur pour fermer la réfutation.

In [9]:
# Exercice 3 — compter les mondes crees sur le cas lobien
def compte_mondes(premisses, borne):
    """Rend le nombre de mondes existants au moment ou le derouleur s'arrete."""
    return None  # TODO etudiant

# Releve attendu : un dictionnaire {borne: nombre_de_mondes} pour borne de 1 a 6,
# obtenu sur CAS["Lob : H <-> []H"]. Conclusion attendue en deux phrases :
# pourquoi aucune borne ne rendra ce cas sat ni unsat, et ce qu'il faudrait
# ajouter au derouleur pour fermer la refutation.
releve = None  # TODO etudiant

print("Exercice 3 — compte_mondes et le releve restent a completer.")

Exercice 3 — compte_mondes et le releve restent a completer.


## À retenir

- La clause de Payor est `P ↔ □(□P→Q)` : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement de celle de l'adversaire. C'est la boîte intérieure qui fait la différence avec la clause löbienne `F(X) ↔ □X(F)`.
- Sur un cadre de Kripke, cette clause se réfute en **deux** mondes contre un autre Payorien, et en **un** contre CooperateBot. Les deux profondeurs sont mesurées et vérifiées par `assert`.
- Contre DefectBot, la réfutation **ne conclut pas** : elle rend un modèle à deux mondes. Ce `sat` est la frontière de l'hypothèse de cohérence, pas une preuve que le bot trahit.
- La route löbienne demande, dans le post, quatre mondes et une auxiliaire `L ↔ (□L→H)`. Ce carnet la **cite** sans la re-mesurer : le dérouleur n'a ni l'axiome de Löb ni la bonne-fondation, et rend `incomplet` sur `H ↔ □H`. Un `incomplet` est un résultat honnête ; le maquiller en `sat` ou en `unsat` ne le serait pas.
- Le tournoi-jouet illustre sans démontrer : il applique une correspondance textuelle là où la logique borne ce qui est **forcé**. Les deux `assert` du tournoi couvrent les seules cases contraintes.

## Vérifiez votre compréhension

1. Pourquoi `□φ` est-il vrai sur un monde sans successeur ? Quelle est la conséquence pour le modèle `sat` de `P vs DefectBot` ?
2. Dans la réfutation `P vs Payorian Q`, à quel monde la contradiction apparaît-elle, et quelles deux formules s'y rencontrent ?
3. Le dérouleur rend `incomplet` sur le cas löbien. Est-ce parce que les prémisses sont cohérentes, contradictoires, ou parce que l'instrument ne sait pas conclure ?
4. Pourquoi un `sat` obtenu sur une formule contenant une disjonction non décomposée ne vaut-il rien ?

<details>
<summary>Éléments de réponse</summary>

1. La boîte est un « pour tout successeur » : le moteur la rend par un `all(...)` sur les successeurs du monde, et un monde terminal n'en a aucun — le `all` porte sur un ensemble vide et vaut vrai. La sortie du mini-test le montre (`[]P en 0 : vrai`) et le commentaire le nomme : « au monde mort (1 sans successeur), []P est vacuement vrai ». C'est ce qui fait tenir le modèle `sat` de `P vs DefectBot` : le monde 1 porte `[]P` et `non Q`, mais comme il ne voit rien, la clause `[]([]P -> Q)` y est vacuement vraie ; la vérification du `sat` le dit — `P` y est vrai par la biconditionnelle, pas par propagation de `[]P`. Le `sat` (modèle à deux mondes) est donc un modèle des prémisses, à la frontière de l'hypothèse de cohérence, pas une preuve que le bot trahit.
2. Au monde 2, et les deux formules qui s'y rencontrent sont `P` et `non P`. Le tableau de la section 3 donne, pour `P vs Payorian Q`, le statut `unsat`, la borne `2` et le détail `contradiction en 2 : P` ; la lecture qui le suit précise le mécanisme : en `w1` on a `[]P` et `non Q` ; le `non Q` attaque la définition de Q, qui ouvre `w2` avec `non P`, tandis que le `[]P` de `w1` place `P` dans `w2`. La contradiction naît entre les deux définitions, dans un monde que chacune atteint par un chemin différent.
3. Parce que l'instrument ne sait pas conclure — le statut ne dit rien de la cohérence des prémisses. Le tableau donne, pour `Lob : H <-> []H`, le statut `incomplet`, la borne `None` et le détail `borne atteinte, obligation ouverte` ; la lecture des statuts est explicite : `incomplet` = « borne atteinte, obligation encore ouverte », « ni cohérent, ni contradictoire : on ne sait pas ». Sur la phrase de Henkin `H <-> []H`, chaque passe ouvre un monde de plus, indéfiniment, et le dérouleur butte sur sa borne. La conclusion du carnet : « ce n'est pas un échec de la réfutation, c'est la limite de l'instrument » — un prouveur de GL fermerait, parce qu'il dispose de l'axiome de Löb et de la bonne-fondation, que ce dérouleur n'a pas.
4. Parce que le `sat` ne vaut que par le modèle qu'on peut exhiber : la vérification reconstruit un `Modele` à partir des mondes saturés et repasse chaque prémisse au moteur de forcing. Or le dérouleur ne décompose ni les disjonctions ni les négations de conjonction : il ne connaît que `¬¬`, `∧`, `→`, `¬(A→B)`, `□`, `¬◊`, `¬□` et `◊`. Une formule qu'il n'a pas décomposée — par exemple `¬(A∧B)`, qui vaut `¬A ∨ ¬B` — reste dans le monde sans être traitée : aucune branche n'est choisie, donc la saturation ne correspond à aucune obligation remplie, et le `sat` rendu « ne voudrait rien dire ». C'est pourquoi les quatre cas de la section 3 ont été choisis sans disjonction, et pourquoi l'exercice 1 demande un `refute_branche` qui explore les deux branches et ne déclare `unsat` que si toutes se ferment.

</details>

## Pour aller plus loin

- Le post d'origine et sa discussion : *Payorian cooperation is easy with Kripke frames*, Alignment Forum — archive du gisement, `GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames.pdf`.
- Dans ce dépôt : `Dynamics-04b-FairBot-Loeb-Lean` pour la route löbienne et sa mécanisation partielle, `Dynamics-05-Agents-Bornes-Python` pour les agents bornés, `Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python` pour le coût du raisonnement et l'Open Problem 3.
- Le moteur de Kripke de la section 2 est une copie pédagogique déclarée de `SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python` : le lire pour la logique modale générale dont ce carnet n'utilise qu'un fragment.